Loading the dataset.

In [21]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)
from sklearn.utils.class_weight import compute_class_weight

sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import build_datasets

In [22]:
IMG_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 32

EPOCHS = 20
LEARNING_RATE = 1e-5

In [23]:
train_ds, val_ds, test_ds, NUM_CLASSES = build_datasets(
    splits_dir="../data/splits",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    augment_train=True,
    use_processed=True,
    processed_dir="../data/processed"
)
print(f"Dataset loaded with {NUM_CLASSES} classes.")


Dataset loaded with 23 classes.


Applied augmentation for the random flip of images, rotation and zooming

In [ ]:
def get_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
    ], name="augmentation")

Model is built using a pretrained ResNet50 architecture, initialized with weights from ImageNet. I

- The majority of the layers are not trainable, to keep a pretrained knowledge, and the top 20 layers are allowed to adapt to the new dataset.
- Batch Normalization layers are kept frozen to maintain stable feature distributions during training
- Global Average Pooling to reduce dimensions
- Dropout was applied to prevent overfitting 
- Dense layer with softmax activation produces class probabilities for the target categories


In [25]:
def build_resnet50_model(input_shape=(224, 224, 3), num_classes=23):
    data_augmentation = get_augmentation()

    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    # Fine-tuning: unfreeze only top layers
    base_model.trainable = True

    for layer in base_model.layers[:-20]:
        layer.trainable = False

    for layer in base_model.layers[-20:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = tf.keras.applications.resnet50.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="model_resnet50")
    return model, base_model



In [26]:
model_1, base_model_1 = build_resnet50_model(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES
)

We have added Callbacks to improve the training process automatically while the model is learning.



- Stop training early
 if the model stops improving (prevents overfitting)
- Adjust learning rate
 make learning slower when needed to improve accuracy
- Save the best model
 keep the best version even if later training gets worse

In [27]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "best_resnet50_augmented.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

In [28]:
model_1.summary()


Model: "model_resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation        │ (None, 224, 224,  │          0 │ input_layer_7[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_6          │ (None, 224, 224)  │          0 │ augmentation[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_7          │ (None, 224, 224)  │          0 │ augmentation[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_8          │ (None, 224, 224)  │          0 │ augmentation[0][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_2 (Stack)     │ (None, 224, 224,  │          0 │ get_item_6[0][0], │
│                     │ 3)                │            │ get_item_7[0][0], │
│                     │                   │            │ get_item_8[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 224, 224,  │          0 │ stack_2[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_2[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 23)        │     47,127 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,634,839 (90.16 MB)

 Trainable params: 8,966,167 (34.20 MB)

 Non-trainable params: 14,668,672 (55.96 MB)

In [29]:
model_1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [30]:
history_1 = model_1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0831 - loss: 3.2101
Epoch 1: val_loss improved from None to 2.95708, saving model to best_resnet50_augmented.keras

Epoch 1: finished saving model to best_resnet50_augmented.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 469s 2s/step - accuracy: 0.0960 - loss: 3.0982 - val_accuracy: 0.1224 - val_loss: 2.9571 - learning_rate: 1.0000e-05
Epoch 2/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1171 - loss: 2.9620
Epoch 2: val_loss improved from 2.95708 to 2.80372, saving model to best_resnet50_augmented.keras

Epoch 2: finished saving model to best_resnet50_augmented.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 457s 2s/step - accuracy: 0.1227 - loss: 2.9287 - val_accuracy: 0.1794 - val_loss: 2.8037 - learning_rate: 1.0000e-05
Epoch 3/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1553 - loss: 2.8471
Epoch 3: val_loss did not improve from 2.80372
292/292 ━━━━━━━━━━━━━━━━━━━━ 460s 2s/step - accuracy: 0.1565 - loss: 2.8280 - val_a

In [31]:
test_loss_ft, test_acc_ft = model_1.evaluate(test_ds, verbose=1)
print(f"Model Test Loss: {test_loss_ft:.4f}")
print(f"Model Test Accuracy: {test_acc_ft:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.2532 - loss: 2.5339
Model Test Loss: 2.5339
Model Test Accuracy: 0.2532


Model 2 - adjusted augmentation

In [32]:
def augmentation_model_2():
    return keras.Sequential([
        layers.RandomCrop(200, 200),
        layers.Resizing(224, 224),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
    ], name="augmentation_model_2")

In [33]:
def build_resnet50_model_2(input_shape=(224, 224, 3), num_classes=23):
    data_augmentation = augmentation_model_2()

    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    # Fine-tuning: unfreeze only top layers
    base_model.trainable = True

    for layer in base_model.layers[:-20]:
        layer.trainable = False

    for layer in base_model.layers[-20:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = tf.keras.applications.resnet50.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="model_resnet50")
    return model, base_model

In [34]:
model_2, base_model_2 = build_resnet50_model_2(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES
)

In [35]:
model_2.summary()

Model: "model_resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_model… │ (None, 224, 224,  │          0 │ input_layer_10[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_9          │ (None, 224, 224)  │          0 │ augmentation_mod… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_10         │ (None, 224, 224)  │          0 │ augmentation_mod… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_11         │ (None, 224, 224)  │          0 │ augmentation_mod… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_3 (Stack)     │ (None, 224, 224,  │          0 │ get_item_9[0][0], │
│                     │ 3)                │            │ get_item_10[0][0… │
│                     │                   │            │ get_item_11[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 224, 224,  │          0 │ stack_3[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_3[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 23)        │     47,127 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,634,839 (90.16 MB)

 Trainable params: 8,966,167 (34.20 MB)

 Non-trainable params: 14,668,672 (55.96 MB)

In [36]:
model_2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [37]:
history_2 = model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0850 - loss: 3.1871
Epoch 1: val_loss did not improve from 2.47576
292/292 ━━━━━━━━━━━━━━━━━━━━ 450s 2s/step - accuracy: 0.0957 - loss: 3.0937 - val_accuracy: 0.1334 - val_loss: 2.9912 - learning_rate: 1.0000e-05
Epoch 2/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1055 - loss: 2.9976
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.

Epoch 2: val_loss did not improve from 2.47576
292/292 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.1140 - loss: 2.9604 - val_accuracy: 0.1454 - val_loss: 2.8852 - learning_rate: 1.0000e-05
Epoch 3/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1317 - loss: 2.8935
Epoch 3: val_loss did not improve from 2.47576
292/292 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.1363 - loss: 2.8731 - val_accuracy: 0.1429 - val_loss: 2.8689 - learning_rate: 5.0000e-06
Epoch 4/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1495 - loss: 2.8272
Epoc